# 04d — Fine-tuning multitarea del D10Sformer

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 4d — Fine-tuning de las heads (Result + Score + MLM auxiliar) sobre el corpus de selecciones

## Objetivo

1. Partir del `best.pt` de Fase 4c (val_loss=2.07, val_acc=0.40 en MLM).
2. Activar **todas las heads**: MLM (aux, λ=0.2) + Result (principal, λ=1.0) + Score (secundario, λ=0.3).
3. Entrenar **10 épocas** sobre `finetune_train.pkl` (8.658 docs, solo selecciones).
4. **LR = 5e-5** (10× menor que pre-train, estándar BERT para fine-tuning).
5. Evaluar contra `val.pkl` y comparar contra los baselines de Fase 1 (LogReg, XGBoost, LightGBM, ELO solo).

## Por qué multitarea

- **MLM auxiliar** evita que las representaciones se desplacen demasiado lejos de las aprendidas en pre-train (regularización).
- **Result head** es la tarea principal: clasificación 3-way (home_win / draw / away_win).
- **Score head** es secundaria (36 clases): le da al modelo señal complementaria sobre la distribución de goles.

Esto es el camino canónico BERT (Devlin et al., 2018 §3.2).

---
## 1. Setup + GPU

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import torch
print(f'torch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('Necesitamos GPU. Runtime → Change runtime type → T4.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import sys, json, pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
PRETRAIN_CKPT = paths.checkpoints_v1 / 'pretrain_5ep' / 'best.pt'

assert PRETRAIN_CKPT.exists(), f'No encuentro {PRETRAIN_CKPT}. ¿Corriste Fase 4c?'
print(f'✓ Pre-train checkpoint: {PRETRAIN_CKPT}  ({PRETRAIN_CKPT.stat().st_size / 1024**2:.1f} MB)')

from data.vocabulary import FootballVocab, RESULT_TOKENS
from data.tokenizer import MatchTokenizer
from data.dataset import MatchDataset
from data.collator import MLMCollator
from models.d10sformer import D10Sformer, D10SformerConfig
from training.trainer import Trainer, TrainerConfig, LossSpec

---
## 2. Cargar vocab + corpus de fine-tuning

In [ ]:
vocab = FootballVocab.load(VOCAB_PATH)
tokenizer = MatchTokenizer(vocab, max_seq_length=80)

with open(CORPUS_DIR / 'finetune_train.pkl', 'rb') as f:
    finetune_docs = pickle.load(f)
with open(CORPUS_DIR / 'val.pkl', 'rb') as f:
    val_docs = pickle.load(f)
with open(CORPUS_DIR / 'test.pkl', 'rb') as f:
    test_docs = pickle.load(f)

print(f'Fine-tune (selecciones train): {len(finetune_docs):,}')
print(f'Val:                           {len(val_docs):,}')
print(f'Test:                          {len(test_docs):,}')

ds_train = MatchDataset(finetune_docs, tokenizer)
ds_val   = MatchDataset(val_docs, tokenizer)
ds_test  = MatchDataset(test_docs, tokenizer)

collator = MLMCollator(vocab, mlm_probability=0.15, seed=42)

BATCH_SIZE = 64
train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, num_workers=0)
val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, num_workers=0)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, num_workers=0)

steps_per_epoch = len(train_loader)
print(f'\nTrain batches/época: {steps_per_epoch}  (batch={BATCH_SIZE})')

---
## 3. Construir modelo + cargar pesos pre-entrenados

In [ ]:
model_config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80, num_segments=8,
    dropout=0.1, attention_dropout=0.1,
    pad_token_id=vocab.encode('[PAD]'),
    tie_mlm_weights=True,
)
model = D10Sformer(model_config)

# Cargar pesos pre-entrenados
ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'✓ Modelo cargado desde paso {ckpt["step"]} (pre-train val_loss={ckpt["best_val_loss"]:.4f})')
print(f'  Params: {model.num_parameters():,}')

---
## 4. Configurar Trainer con LossSpec multitarea

In [ ]:
EPOCHS = 10
MAX_STEPS = steps_per_epoch * EPOCHS
print(f'EPOCHS = {EPOCHS}')
print(f'Pasos totales = {MAX_STEPS}')

trainer_config = TrainerConfig(
    lr=5e-5,                # 10× menor que pre-train
    weight_decay=0.01,
    grad_clip_norm=1.0,
    warmup_ratio=0.1,
    max_steps=MAX_STEPS,
    mixed_precision=True,
    log_every=25,
    eval_every=100,
    save_every=300,
    save_best=True,
    output_dir=str(CKPT_DIR),
    run_name='finetune_10ep',
    seed=42,
)
loss_spec = LossSpec(
    use_mlm=True,    lambda_mlm=0.2,     # auxiliar, regularizador
    use_result=True, lambda_result=1.0,  # tarea principal
    use_score=True,  lambda_score=0.3,   # secundaria
)
print(f'\nLossSpec: λ_mlm={loss_spec.lambda_mlm}, λ_result={loss_spec.lambda_result}, λ_score={loss_spec.lambda_score}')

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=trainer_config,
    loss_spec=loss_spec,
)
print(f'Device: {trainer.device}  AMP: {trainer.use_amp}')
print(f'Output: {trainer.output_dir}')

---
## 5. Fine-tuning (10 épocas)

In [ ]:
import time
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Fine-tuning completado en {elapsed/60:.1f} min ({MAX_STEPS/elapsed:.1f} step/s)')
print(f'  Best val_loss: {trainer.best_val_loss:.4f}')
print(f'  Best checkpoint: {trainer.output_dir / "best.pt"}')

---
## 6. Curvas de fine-tuning

In [ ]:
rows = [json.loads(l) for l in trainer.log_path.read_text().splitlines() if l.strip()]
train_rows = [r for r in rows if 'phase' not in r]
eval_rows  = [r for r in rows if r.get('phase') == 'eval']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot([r['step'] for r in train_rows], [r['loss'] for r in train_rows], label='train (total)')
if eval_rows:
    axes[0, 0].plot([r['step'] for r in eval_rows], [r['val_loss'] for r in eval_rows], 'o-', label='val (total)', linewidth=2)
axes[0, 0].set_xlabel('step'); axes[0, 0].set_ylabel('loss'); axes[0, 0].set_title('Total multitask loss')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot([r['step'] for r in train_rows], [r['mlm_perplexity'] for r in train_rows], label='train')
if eval_rows:
    axes[0, 1].plot([r['step'] for r in eval_rows], [r['val_mlm_perplexity'] for r in eval_rows], 'o-', label='val', linewidth=2)
axes[0, 1].set_xlabel('step'); axes[0, 1].set_ylabel('perplexity (MLM)')
axes[0, 1].set_title('MLM perplexity (auxiliar)')
axes[0, 1].set_yscale('log'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot([r['step'] for r in train_rows], [r['mlm_acc'] for r in train_rows], label='train')
if eval_rows:
    axes[1, 0].plot([r['step'] for r in eval_rows], [r['val_mlm_acc'] for r in eval_rows], 'o-', label='val', linewidth=2)
axes[1, 0].set_xlabel('step'); axes[1, 0].set_ylabel('top-1 acc'); axes[1, 0].set_title('MLM accuracy')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot([r['step'] for r in train_rows], [r['lr'] for r in train_rows], color='purple')
for ep in range(1, EPOCHS):
    axes[1, 1].axvline(ep * steps_per_epoch, color='gray', linestyle=':', alpha=0.4)
axes[1, 1].set_xlabel('step'); axes[1, 1].set_ylabel('lr'); axes[1, 1].set_title('LR schedule (5e-5 + cosine)')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(trainer.output_dir / 'training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 7. Evaluación detallada sobre VAL (Result head)

Recargamos el mejor checkpoint (`best.pt`) y computamos:
- Accuracy
- Log-loss (CE)
- Brier score multiclase
- ECE (Expected Calibration Error)
- Distribución de probabilidades + confusion matrix

In [ ]:
from eval.metrics import (
    multiclass_log_loss, multiclass_brier_score,
    expected_calibration_error, evaluate_all,
)

# Cargar el best.pt del fine-tuning
best_path = trainer.output_dir / 'best.pt'
best_ckpt = torch.load(best_path, map_location=trainer.device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()
print(f'✓ Cargado best (step={best_ckpt["step"]}, val_loss={best_ckpt["best_val_loss"]:.4f})')

@torch.no_grad()
def predict_result_probs(loader):
    """Devuelve (y_true, y_prob) sobre todo el loader. Filtra entries sin target."""
    all_probs = []
    all_true  = []
    for batch in loader:
        batch = batch.to(trainer.device)
        out = model(batch.token_ids, batch.segment_ids, attention_mask=batch.attention_mask)
        probs = F.softmax(out['result_logits'], dim=-1).cpu().numpy()  # (B, 3)
        true_global_ids = batch.result_labels.cpu().numpy()
        # Filtrar -100 y mapear ids globales a [0,1,2]
        for p, gid in zip(probs, true_global_ids):
            if gid == -100:
                continue
            # Mapeo: el orden de las heads result_logits es {0=HOME, 1=DRAW, 2=AWAY}
            # pero target_result_id viene del vocab global. Tenemos que mapear.
            tok = vocab.decode(int(gid))
            if tok == 'RESULT_HOME_WIN':   y = 0
            elif tok == 'RESULT_DRAW':     y = 1
            elif tok == 'RESULT_AWAY_WIN': y = 2
            else: continue
            all_probs.append(p)
            all_true.append(y)
    return np.array(all_true), np.array(all_probs)

y_val, p_val = predict_result_probs(val_loader)
y_test, p_test = predict_result_probs(test_loader)
print(f'Val: y_true {y_val.shape}, y_prob {p_val.shape}')
print(f'Test: y_true {y_test.shape}, y_prob {p_test.shape}')

In [ ]:
metrics_val = evaluate_all(y_val, p_val)
metrics_test = evaluate_all(y_test, p_test)

print('=== D10Sformer fine-tuneado ===')
print(f'{"métrica":<20} {"VAL":<12} {"TEST":<12}')
for k in ['log_loss', 'brier', 'ece', 'accuracy']:
    print(f'{k:<20} {metrics_val[k]:<12.4f} {metrics_test[k]:<12.4f}')

In [ ]:
# Confusion matrix
from collections import Counter
preds_val = p_val.argmax(axis=1)
cm = np.zeros((3, 3), dtype=int)
for yt, yp in zip(y_val, preds_val):
    cm[yt, yp] += 1

labels = ['home_win', 'draw', 'away_win']
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() * 0.6 else 'black')
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion matrix (VAL, n={len(y_val)})')
plt.colorbar(im)
plt.tight_layout()
plt.show()

print(f'\nDistribución VAL: {Counter(y_val.tolist())}')
print(f'Predicciones VAL: {Counter(preds_val.tolist())}')

---
## 8. Comparación contra baselines de Fase 1

In [ ]:
# Tabla de comparación (los baselines vienen del paper / Fase 1)
# Si tenés guardadas las métricas exactas en un JSON, lo cargamos. Si no, las dejo hardcodeadas
# del notebook 01_baselines:
baselines_fase1 = {
    'LogReg':         {'log_loss': 0.8610, 'brier': 0.5071, 'ece': 0.0236, 'accuracy': 0.6005},
    'XGBoost':        {'log_loss': 0.8663, 'brier': 0.5105, 'ece': 0.0182, 'accuracy': 0.6026},
    'LightGBM':       {'log_loss': 0.8744, 'brier': 0.5135, 'ece': 0.0250, 'accuracy': 0.5990},
    'ELO solo':       {'log_loss': 1.0102, 'brier': 0.6101, 'ece': 0.0890, 'accuracy': 0.5510},
    'Uniforme':       {'log_loss': 1.0986, 'brier': 0.6667, 'ece': 0.0, 'accuracy': 0.4500},
}

comparison = baselines_fase1.copy()
comparison['D10Sformer (val)']  = {k: metrics_val[k] for k in ['log_loss', 'brier', 'ece', 'accuracy']}
comparison['D10Sformer (test)'] = {k: metrics_test[k] for k in ['log_loss', 'brier', 'ece', 'accuracy']}

import pandas as pd
df_cmp = pd.DataFrame(comparison).T
print('=== Comparación D10Sformer vs Baselines (Fase 1) ===')
print(df_cmp.to_string(float_format=lambda x: f'{x:.4f}'))

# Highlight: ¿le ganamos al mejor baseline en log_loss y accuracy?
best_baseline_ll = min(baselines_fase1[k]['log_loss'] for k in baselines_fase1)
best_baseline_acc = max(baselines_fase1[k]['accuracy'] for k in baselines_fase1)

print(f'\nMejor baseline log_loss:  {best_baseline_ll:.4f}')
print(f'D10Sformer val log_loss:  {metrics_val["log_loss"]:.4f} ({"GANA" if metrics_val["log_loss"] < best_baseline_ll else "pierde"})')
print(f'D10Sformer test log_loss: {metrics_test["log_loss"]:.4f} ({"GANA" if metrics_test["log_loss"] < best_baseline_ll else "pierde"})')
print(f'\nMejor baseline accuracy:  {best_baseline_acc:.4f}')
print(f'D10Sformer val accuracy:  {metrics_val["accuracy"]:.4f} ({"GANA" if metrics_val["accuracy"] > best_baseline_acc else "pierde"})')
print(f'D10Sformer test accuracy: {metrics_test["accuracy"]:.4f} ({"GANA" if metrics_test["accuracy"] > best_baseline_acc else "pierde"})')

---
## 9. Conclusiones de Fase 4d

- [ ] Tiempo de fine-tuning: _____ min
- [ ] Loss inicial / final: _____ / _____
- [ ] Best val_loss: _____ (paso _____)
- [ ] Métricas finales sobre TEST:
  - log_loss: _____
  - brier: _____
  - ece: _____
  - accuracy: _____
- [ ] ¿Le ganamos al mejor baseline en log_loss? _____
- [ ] ¿Le ganamos en accuracy? _____
- [ ] ¿En ECE? (calibración) _____
- [ ] ¿La confusion matrix muestra que el modelo predice las 3 clases o se concentra en 1-2? _____

**Next:** Fase 5 — evaluación profunda + reliability diagrams + análisis de errores por bucket de ELO + ablation studies (sin pre-train, sin features, etc.).